# Analysis

## Setting parameters

In [8]:
%load_ext autoreload
%autoreload 2

In [9]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

In [44]:
# data_sets = [  # actual set for QR
#     'australian',
#      'automobile',
#      'bands',
#      'bupa',
#      'cleveland',
#      'contraceptive',
#      'crx',
#      'dermatology',
#      'ecoli',
#      'german',
#      'glass',
#      'haberman',
#      'heart',
#      'ionosphere',
#      'mammographic',
#      'pima',
#      'saheart',
#      # 'sonar',  # 70 features!
#      'spectfheart',
#      'vehicle',
#      'vowel',
#      'wine',
#      'winequality-red',
#      'wisconsin',
#      'yeast'
# ]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
control = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    'ecoli',
    'glass',
    'heart',
    'ionosphere',
    'pima',
    'spectfheart',
    'winequality-red',
    'wine',
    'wisconsin',
    'yeast'
]

In [45]:
data_sets = control

In [46]:
len(data_sets)

13

In [47]:
data_base = Path.cwd() / 'keel-data'
include_sets = data_sets
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd()

## Loading MODLEM results

In [48]:
weka_folder = Path.cwd() / 'weka_rules_files'
name = 'modlem'
file = 'weka-balanced-accuracy.csv'  # 'weka-accuracy.csv'

In [49]:
scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]

In [50]:
rules[name] = {}
attributes[name] = {}
for file in weka_folder.iterdir():
    if file.name[-3:] == 'txt':
        short_name = file.name[:-4]
        with open(file, 'r') as f:
            line = f.readline()
            nrs = []
            while len(line) > 4:
                nrs.append(line.count('&') + 1)
                line = f.readline()
        rules[name][short_name] = len(nrs)
        attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [51]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [52]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=include_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=include_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=include_sets
    )

/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:518: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/numpy/lib/function_base.py:518: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## Adding results for non rule based models


In [53]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [54]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=include_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [55]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in include_sets}

## NORI Models

In [56]:
nori_models = {
    'non-overlap': Path.cwd() / 'non-overlap-rules',
    'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    'lower-check' : Path.cwd() / 'lower-approx-og-implement-test'
}

In [57]:
for model, path in nori_models.items():
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
            path,
            include=data_sets
        )
    attributes[model] = count_all_attributes(
        Path.cwd() / 'non-overlap-rules',
        include=include_sets,
        counter='{'
    )

In [58]:
scores.keys()

dict_keys(['modlem', 'qr', 'lin-owa', 'trun-lin-owa', 'trun-exp-owa', 'avg-qr', 'avg-lin-owa', 'prune-qr', 'prune-lin-owa', 'prune-trun-lin-owa', 'prune-trun-exp-owa', 'prune-avg-qr', 'prune-avg-lin-owa', 'naive-bayes', 'svm', 'tree', 'frnn', 'non-overlap', 'nori', 'lower-nori', 'lower-check'])

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [59]:
 names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    6: [
        'nori',
        'lower-nori',
        'lower-check',
        'modlem',
        'qr',
    ]
}

In [60]:
nr = 6

In [61]:
main_folder = 'balanced_acc'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [62]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [63]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]

In [64]:
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)

In [65]:
table_acc

,nori,lower-nori,lower-check,modlem,qr
australian,0.789347,0.805928,0.798701,0.861,0.865348
bupa,0.58792,0.634599,0.584774,0.6525,0.5
cleveland,0.282424,0.308859,0.315693,0.2764,0.208347
ecoli,0.634576,0.645814,0.719516,0.512,0.17585
glass,0.642698,0.642619,0.669524,0.518333,0.31381
heart,0.775,0.728333,0.763333,0.7605,0.785833
ionosphere,0.88512,0.916868,0.915895,0.8905,0.658974
pima,0.640461,0.674594,0.654653,0.6985,0.502775
spectfheart,0.562641,0.569913,0.690152,0.6015,0.559156
winequality-red,0.31292,0.36881,0.335618,0.3385,0.182208


In [147]:
bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_84935/3326315922.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [135]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)

In [136]:
table_rule

,nori,lower-nori,modlem,qr
australian,102.4,100.4,121,732.2
bupa,101.3,103.1,103,332.4
cleveland,104.3,109.2,95,331.1
ecoli,75.5,76.1,56,315.0
glass,60.3,61.6,50,225.7
heart,45.0,47.8,62,303.7
ionosphere,27.0,27.2,30,460.5
pima,165.1,169.4,191,824.6
spectfheart,37.6,37.0,55,126.0
wine,11.9,10.6,13,211.2


In [148]:
bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.1f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_84935/1993066615.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [142]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)

In [144]:
table_attribute

,nori,lower-nori,modlem,qr
australian,13.878353,13.878353,2.355372,8.04978
bupa,10.589309,10.589309,2.194175,4.40226
cleveland,14.886441,14.886441,2.589474,8.08194
ecoli,10.186761,10.186761,2.339286,3.816832
glass,10.219248,10.219248,2.1,5.343654
heart,12.91109,12.91109,2.209677,7.580314
ionosphere,11.149098,11.149098,1.533333,11.653025
pima,12.737907,12.737907,2.073298,5.687857
spectfheart,15.612316,15.612316,1.745455,1.171367
wine,9.611248,9.611248,1.461538,6.969346


In [149]:
bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min'), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_84935/3339941083.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)
